# 00 — Clean and merge predictions

Merges **480 raw CSV files per model** into one clean file for every model × approach × history configuration.

- Models: Qwen, LLaVA, InternVL
- Approaches: `frame → appfr`, `sensor → appod`
- Histories: H02, H04, ..., H32
- Output: 96 clean prediction files (32 per model)
- Frame level metric e.g., correct binary, msp, pcs, DG, entropy etc. 


In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from IPython.display import display

ANALYSIS = Path.cwd().parent if Path.cwd().name == "Codes" else Path.cwd()
RAW = ANALYSIS / "Model_Pred"
CLEAN = ANALYSIS / "Outputs" / "clean"

MODELS = ["qwen", "llava", "internvl"]
HISTORIES = range(2, 33, 2)
MODE_TO_APPROACH = {"frame": "appfr", "sensor": "appod"}
LABELS = ["safe", "potentially_unsafe", "unsafe"]
PROB_COLS = ["prob_safe", "prob_potentially_unsafe", "prob_unsafe"]
PROB_TO_LABEL = dict(zip(PROB_COLS, LABELS))
EXPECTED_GT = {"safe": 15273, "potentially_unsafe": 1378, "unsafe": 211}
FILE_PATTERN = re.compile(r"_h(\d+)_(frame|sensor)\.csv$")

# Accept the column names used by all three model outputs.
ALIASES = {
    "bag_name": ["bag_name"],
    "frame_id": ["frame_id"],
    "frame_time_s": ["frame_time_s"],
    "ground_truth": ["gt_state", "ground_truth", "ground_truth_label"],
    "model_saved_label": ["predicted_state", "predicted_label"],
    "prob_safe": ["p_safe", "prob_safe"],
    "prob_potentially_unsafe": ["p_potentially_unsafe", "prob_potentially_unsafe"],
    "prob_unsafe": ["p_unsafe", "prob_unsafe"],
}

FINAL_COLS = [
    "bag_name", "frame_id", "frame_time_s", "ground_truth",
    "model_saved_label", "predicted_label", "prediction_corrected",
    *PROB_COLS, "msp", "pcs", "entropy", "normalized_entropy",
    "deep_gini", "correct_binary"
]

print("Analysis folder:", ANALYSIS)

In [ ]:
def clean_labels(series):
    return (series.astype(str).str.strip().str.lower().replace({
        "potentially unsafe": "potentially_unsafe",
        "potentially-unsafe": "potentially_unsafe",
        "potentiallyunsafe": "potentially_unsafe",
    }))


def read_prediction(file):
    df = pd.read_csv(file)
    rename = {}
    for target, choices in ALIASES.items():
        source = next((c for c in choices if c in df.columns), None)
        if source is None:
            raise ValueError(f"{file}: missing {target}; columns={list(df.columns)}")
        rename[source] = target
    return df[list(rename)].rename(columns=rename)


def add_frame_metrics(df):
    df["ground_truth"] = clean_labels(df["ground_truth"])
    df["model_saved_label"] = clean_labels(df["model_saved_label"])
    df[PROB_COLS] = df[PROB_COLS].apply(pd.to_numeric, errors="coerce")

    probs = df[PROB_COLS].to_numpy(float)
    df["predicted_label"] = df[PROB_COLS].idxmax(axis=1).map(PROB_TO_LABEL)
    df["prediction_corrected"] = df["model_saved_label"] != df["predicted_label"]
    df["correct_binary"] = (df["ground_truth"] == df["predicted_label"]).astype(int)
    df["msp"] = probs.max(axis=1)
    sorted_probs = np.sort(probs, axis=1)
    df["pcs"] = sorted_probs[:, -1] - sorted_probs[:, -2]
    df["entropy"] = -(probs * np.log(probs + 1e-12)).sum(axis=1)
    df["normalized_entropy"] = df["entropy"] / np.log(len(LABELS))
    df["deep_gini"] = 1 - np.square(probs).sum(axis=1)
    return df


def validate(df, model, history, mode):
    name = f"{model} H{history:02d} {mode}"
    assert len(df) == 16862, f"{name}: {len(df)} rows"
    assert df["bag_name"].nunique() == 15, f"{name}: wrong bag count"
    assert not df.duplicated(["bag_name", "frame_id"]).any(), f"{name}: duplicate frames"
    assert df["ground_truth"].value_counts().to_dict() == EXPECTED_GT, f"{name}: wrong GT"
    assert set(df["ground_truth"]) <= set(LABELS), f"{name}: unknown GT label"
    assert np.isfinite(df[PROB_COLS].to_numpy()).all(), f"{name}: invalid probability"
    assert ((df[PROB_COLS] >= 0) & (df[PROB_COLS] <= 1)).all().all(), f"{name}: probability outside [0,1]"
    assert np.isclose(df[PROB_COLS].sum(axis=1), 1, atol=1e-5).all(), f"{name}: probabilities do not sum to 1"
    assert not df[FINAL_COLS].isna().any().any(), f"{name}: missing values"

In [ ]:
# Discover files before writing anything.
groups = {}
inventory = []

for model in MODELS:
    model_files = []
    for file in sorted((RAW / model).rglob("*.csv")):
        match = FILE_PATTERN.search(file.name)
        if match:
            history, mode = int(match.group(1)), match.group(2)
            groups.setdefault((model, history, mode), []).append(file)
            model_files.append(file)

    inventory.append({"model": model, "raw_files": len(model_files)})
    assert len(model_files) == 480, f"{model}: expected 480 files, found {len(model_files)}"

for model in MODELS:
    for history in HISTORIES:
        for mode in MODE_TO_APPROACH:
            assert len(groups.get((model, history, mode), [])) == 15, (
                f"{model} H{history:02d} {mode}: expected 15 bag files"
            )

display(pd.DataFrame(inventory))
print("Ready: 1,440 raw files in 96 configurations.")

In [ ]:
summary_rows, correction_rows = [], []

for model in MODELS:
    model_out = CLEAN / model
    model_out.mkdir(parents=True, exist_ok=True)

    for history in HISTORIES:
        for mode, approach in MODE_TO_APPROACH.items():
            files = groups[(model, history, mode)]
            df = pd.concat([read_prediction(f) for f in files], ignore_index=True)
            df = add_frame_metrics(df).sort_values(["bag_name", "frame_id"]).reset_index(drop=True)
            validate(df, model, history, mode)

            output = model_out / f"{model}_{approach}_h{history:02d}.csv"
            df[FINAL_COLS].to_csv(output, index=False)

            changed = df.loc[df["prediction_corrected"], [
                "bag_name", "frame_id", "ground_truth", "model_saved_label",
                "predicted_label", *PROB_COLS
            ]].copy()
            if not changed.empty:
                changed.insert(0, "history", f"H{history:02d}")
                changed.insert(0, "approach", approach)
                changed.insert(0, "model", model)
                correction_rows.append(changed)

            counts = df["predicted_label"].value_counts()
            summary_rows.append({
                "model": model, "approach": approach, "history": f"H{history:02d}",
                "source_files": len(files), "bags": df["bag_name"].nunique(),
                "rows": len(df), "corrected_saved_labels": int(df["prediction_corrected"].sum()),
                "pred_safe": int(counts.get("safe", 0)),
                "pred_potentially_unsafe": int(counts.get("potentially_unsafe", 0)),
                "pred_unsafe": int(counts.get("unsafe", 0)), "output_file": output.name,
            })
            print(f"Saved {model} {approach} H{history:02d}")

summary = pd.DataFrame(summary_rows)
corrections = pd.concat(correction_rows, ignore_index=True) if correction_rows else pd.DataFrame()
summary.to_csv(CLEAN / "merge_summary.csv", index=False)
corrections.to_csv(CLEAN / "label_corrections.csv", index=False)

display(summary.groupby("model").agg(clean_files=("output_file", "count"), rows_per_file=("rows", "first"), label_corrections=("corrected_saved_labels", "sum")))
print(f"\nFinished: {len(summary)} clean files saved in {CLEAN}")